In [1]:
import colorsys
import hashlib
import math
import os
from pathlib import Path

import h3pandas
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm
import pydeck as pdk


from lib.models.geo_model_net import ResLayer

2025-01-25 14:03:19.753384: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-25 14:03:19.753413: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-25 14:03:19.754368: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-25 14:03:19.758763: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-25 14:03:20.300957: W tensorflow/compiler/tf2

In [2]:
tax_2_17 = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20240929050006-aka-2.17/taxonomy.csv"
)
sinr_threshold = pd.read_csv(
    "/home/alex/inatVisionApi/v2_17_sinr_coords_elev_two/thresholds.csv"
).set_index("taxon_id")
grid_threshold = pd.read_csv(
    "/home/alex/inatVisionApi/v2_17_fast/thresholds.csv"
).set_index("taxon_id")

In [4]:
grid_threshold.head(3)

,Unnamed: 0,thres,area
taxon_id,,,
129726,0,0.001032,1.089634e+07
48272,1,0.001083,1.215681e+07
81614,2,0.000146,9.782808e+06


In [5]:
sinr_threshold.head(3)

,Unnamed: 0,thres,area
taxon_id,,,
129726,0,0.002316,8.164732e+06
48272,1,0.006277,7.422967e+06
81614,2,0.001612,3.839832e+06


In [206]:
class TFGeoPriorModelElev:

    def __init__(self, model_path):
        # initialize the geo model for inference
        tf.config.set_visible_devices([], "GPU")
        visible_devices = tf.config.get_visible_devices()
        for device in visible_devices:
            assert device.device_type != "GPU"
        self.gpmodel = tf.keras.models.load_model(
            model_path,
            custom_objects={"ResLayer": ResLayer},
            compile=False
        )

    def predict(self, latitude, longitude, elevation):
        encoded_loc = TFGeoPriorModelElev.encode_loc([latitude], [longitude], [elevation])
        return self.gpmodel(tf.convert_to_tensor(
            tf.expand_dims(encoded_loc[0], axis=0)
        ), training=False)[0]

    def features_for_one_class_elevation(self, latitude, longitude, elevation):
        """Evalutes the model for a single class and multiple locations

        Args:
            latitude (list): A list of latitudes
            longitude (list): A list of longitudes (same length as latitude)
            elevation (list): A list of elevations (same length as latitude)
            class_of_interest (int): The single class to eval

        Returns:
            numpy array: scores for class of interest at each location
        """

        encoded_loc = TFGeoPriorModelElev.encode_loc(latitude, longitude, elevation)
        loc_emb = self.gpmodel.layers[0](encoded_loc)

        # res layers - feature extraction
        x = self.gpmodel.layers[1](loc_emb)
        x = self.gpmodel.layers[2](x)
        x = self.gpmodel.layers[3](x)
        x = self.gpmodel.layers[4](x)

        return x

    def eval_one_class_elevation_from_features(self, features, class_of_interest):
        # process just the one class
        return tf.keras.activations.sigmoid(
            tf.matmul(
                tf.expand_dims(self.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
                features,
                transpose_b=True
            )
        ).numpy()

    @staticmethod
    def encode_loc(latitude, longitude, elevation):
        latitude = np.array(latitude)
        longitude = np.array(longitude)
        elevation = np.array(elevation)
        elevation = elevation.astype("float32")
        grid_lon = longitude.astype("float32") / 180.0
        grid_lat = latitude.astype("float32") / 90.0

        elevation[elevation > 0] = elevation[elevation > 0] / 6574.0
        elevation[elevation < 0] = elevation[elevation < 0] / 32768.0
        norm_elev = elevation

        # if np.isscalar(grid_lon):
        #     grid_lon = np.array([grid_lon])
        # if np.isscalar(grid_lat):
        #     grid_lat = np.array([grid_lat])
        # if np.isscalar(norm_elev):
        #     norm_elev = np.array([norm_elev])

        norm_loc = tf.stack([grid_lon, grid_lat], axis=1)

        encoded_loc = tf.concat([
            tf.sin(norm_loc * math.pi),
            tf.cos(norm_loc * math.pi),
            tf.expand_dims(norm_elev, axis=1),

        ], axis=1)
        return encoded_loc

In [9]:
gp_model_2_17_sinr = TFGeoPriorModelElev(
    "/home/alex/inatGeoModelTraining/saved_model_2_17_ds_sinr_approach_coords_elev.h5"
)
gp_model_2_17_grid = TFGeoPriorModelElev(
    "/home/alex/export_eval_models/model_exports/tf_geoprior_2_17_r6_elevation.h5"
)

In [10]:
#!ls /home/alex/inatGeoModelTraining

In [11]:
geo_elevation_cells = pd.read_csv(
    "/home/alex/elevation_r4_5m.csv"
).set_index("h3_04").sort_index().h3.h3_to_geo()
geo_elevation_cells["lat"] = geo_elevation_cells.geometry.y
geo_elevation_cells["lng"] = geo_elevation_cells.geometry.x
_ = geo_elevation_cells.pop("geometry")

In [12]:
geo_elevation_cells.describe()

,elevation,lat,lng
count,288122.000000,2.881220e+05,288122.000000
mean,-23005.145876,2.020244e-16,-0.371932
std,15009.330107,3.913723e+01,104.039630
min,-32768.000000,-8.995860e+01,-179.997544
25%,-32768.000000,-3.004676e+01,-90.086247
50%,-32768.000000,1.987776e-15,0.000672
75%,71.461197,3.004676e+01,89.912716
max,5705.629630,8.995860e+01,179.998888


In [13]:
geo_model_features_grid = gp_model_2_17_grid.features_for_one_class_elevation(
    latitude=list(geo_elevation_cells.lat),
    longitude=list(geo_elevation_cells.lng),
    elevation=list(geo_elevation_cells.elevation)
)

In [186]:
TARGET_TAXON_ID = 51315

In [187]:
def h3_04_geo_results_for_taxon(taxon_id, model, geo_model_features):
    taxon = tax_2_17[tax_2_17.taxon_id==taxon_id].iloc[0]
    geo_scores = model.eval_one_class_elevation_from_features(
        geo_model_features, taxon.spatial_class_id
    )
    
    return geo_scores

In [188]:
geo_scores = h3_04_geo_results_for_taxon(
    TARGET_TAXON_ID, gp_model_2_17_grid, geo_model_features_grid
)

In [189]:
threshold = grid_threshold.loc[TARGET_TAXON_ID].thres
threshold

0.070826694

In [190]:
geo_scores.max()

0.75721276

In [191]:
geo_score_cells_grid = geo_elevation_cells.copy()
geo_score_cells_grid["geo_score"] = tf.squeeze(geo_scores).numpy()

lower_bound_score = np.array([0.0001, threshold / 10]).min()

geo_score_cells_grid = geo_score_cells_grid[geo_score_cells_grid.geo_score>lower_bound_score]
geo_score_cells_grid = geo_score_cells_grid.reset_index()
len(geo_score_cells_grid)

926

In [192]:
min_score = geo_score_cells_grid.geo_score.min()
max_score = geo_score_cells_grid.geo_score.max()
geo_score_cells_grid["scaled_geo_score"] = (
    np.log10(geo_score_cells_grid["geo_score"]) - math.log10(min_score)) / (math.log10(max_score) - math.log10(min_score)
)

def map_range(value, old_min, old_max, new_min, new_max):
    return (value - old_min) / (old_max - old_min) * (new_max - new_min) + new_min

geo_score_cells_grid["scaled_geo_score_two"] = map_range(
    np.log10(geo_score_cells_grid["geo_score"]),
    np.log10(min_score),
    np.log10(max_score),
    0.,
    1.
)

In [193]:
geo_score_cells_grid["hue"] = 211.0 / 255.0
geo_score_cells_grid["sat"] = geo_score_cells_grid["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.41, 0.9)
)
geo_score_cells_grid["val"] = geo_score_cells_grid["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.4, 0.85)
)
geo_score_cells_grid["alpha"] = geo_score_cells_grid["scaled_geo_score"]

# map_range(
#     geo_score_cells_grid["scaled_geo_score"],
#     min_score,
#     max_score,
#     0.0,
#     1.0
# )
# geo_score_cells_grid["sat"] = geo_score_cells_grid["scaled_geo_score"].apply(lambda x: map_range(x, 0.5, 1., 0.41, 0.90) if x > 0.5 else 0.41)
# geo_score_cells_grid["val"] = geo_score_cells_grid["scaled_geo_score"].apply(lambda x: map_range(x, 0.5, 1., 1., 0.85) if x > 0.5 else 1.0)
# geo_score_cells_grid["alpha"] = geo_score_cells_grid["scaled_geo_score"].apply(lambda x: 1 if x > 0.5 else map_range(x, 0., 0.5, 0., 1.))

geo_score_cells_grid["red"] = geo_score_cells_grid.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[0], axis=1)
geo_score_cells_grid["grn"] = geo_score_cells_grid.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[1], axis=1)
geo_score_cells_grid["blu"] = geo_score_cells_grid.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[2], axis=1)

In [183]:
geo_score_cells_grid

,h3_04,elevation,lat,lng,geo_score,scaled_geo_score,hue,sat,val,alpha,red,grn,blu,scaled_geo_score_two
0,841181bffffffff,165.914286,55.423368,37.799476,0.000146,0.041122,0.827451,0.430150,0.418505,0.041122,0.412151,0.238485,0.418505,0.041122
1,8411aa5ffffffff,186.558824,55.593005,37.199516,0.000263,0.105454,0.827451,0.461672,0.447454,0.105454,0.440163,0.240877,0.447454,0.105454
2,8411aa7ffffffff,156.583333,55.831463,37.756293,0.000219,0.085407,0.827451,0.451849,0.438433,0.085407,0.431441,0.240327,0.438433,0.085407
3,8411b65ffffffff,122.242424,51.317048,32.317810,0.000102,0.001604,0.827451,0.410786,0.400722,0.001604,0.394912,0.236111,0.400722,0.001604
4,8412805ffffffff,694.892857,53.840538,-128.527806,0.000116,0.016552,0.827451,0.418111,0.407448,0.016552,0.401436,0.237090,0.407448,0.016552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4458,84da9c9ffffffff,-20677.366667,-43.641779,169.237630,0.000120,0.019925,0.827451,0.419763,0.408966,0.019925,0.402907,0.237297,0.408966,0.019925
4459,84dab31ffffffff,-30577.133333,-44.091060,172.771501,0.000147,0.042009,0.827451,0.430584,0.418904,0.042009,0.412538,0.238530,0.418904,0.042009
4460,84dab33ffffffff,-29489.200000,-44.145637,172.199683,0.000157,0.048911,0.827451,0.433966,0.422010,0.048911,0.415546,0.238872,0.422010,0.048911
4461,84dab35ffffffff,-10412.322581,-43.700285,173.008248,0.000529,0.182045,0.827451,0.499202,0.481920,0.182045,0.473429,0.241345,0.481920,0.182045


In [194]:
view_state_grid = pdk.ViewState(
    latitude=37.7749295, 
    longitude=-122.4194155, 
    zoom=3, 
    bearing=0, 
    pitch=0
)
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    geo_score_cells_grid,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="[red * 255 * 0.3, grn * 255, blu * 255, alpha * 255]",
    get_line_color=[255, 100, 100],
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[layer_grid], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
r_grid.show()

# ok let's try with the sinr map

In [195]:
geo_elevation_cells = pd.read_csv(
    "/home/alex/elevation_r4_5m.csv"
).set_index("h3_04").sort_index().h3.h3_to_geo()
geo_elevation_cells["lat"] = geo_elevation_cells.geometry.y
geo_elevation_cells["lng"] = geo_elevation_cells.geometry.x
_ = geo_elevation_cells.pop("geometry")

In [196]:
geo_model_features_sinr = gp_model_2_17_sinr.features_for_one_class_elevation(
    latitude=list(geo_elevation_cells.lat),
    longitude=list(geo_elevation_cells.lng),
    elevation=list(geo_elevation_cells.elevation)
)

In [197]:
geo_scores = h3_04_geo_results_for_taxon(
    TARGET_TAXON_ID, gp_model_2_17_sinr, geo_model_features_sinr
)

In [198]:
geo_score_cells_sinr = geo_elevation_cells.copy()
geo_score_cells_sinr["geo_score"] = tf.squeeze(geo_scores).numpy()
lower_bound_score = np.array([0.0001, sinr_threshold.loc[TARGET_TAXON_ID].thres / 10]).min()

geo_score_cells_sinr = geo_score_cells_sinr[geo_score_cells_sinr.geo_score>lower_bound_score]
geo_score_cells_sinr = geo_score_cells_sinr.reset_index()
len(geo_score_cells_sinr)

min_score = geo_score_cells_sinr.geo_score.min()
max_score = geo_score_cells_sinr.geo_score.max()
geo_score_cells_sinr["scaled_geo_score"] = (np.log10(geo_score_cells_sinr["geo_score"]) - math.log10(min_score)) / \
                    (math.log10(max_score) - math.log10(min_score))

In [199]:
geo_score_cells_sinr["hue"] = 211.0 / 255.0
geo_score_cells_sinr["sat"] = geo_score_cells_sinr["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.41, 0.9)
)
geo_score_cells_sinr["val"] = geo_score_cells_sinr["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.4, 0.85)
)
geo_score_cells_sinr["alpha"] = geo_score_cells_sinr["scaled_geo_score"]

# geo_score_cells_sinr["hue"] = 211.0 / 255.0
# geo_score_cells_sinr["sat"] = geo_score_cells_sinr["scaled_geo_score"].apply(lambda x: map_range(x, 0., 1., 0.41, 0.90))
# geo_score_cells_sinr["val"] = geo_score_cells_sinr["scaled_geo_score"].apply(lambda x: map_range(x, 0., 1., 1., 0.85))
# geo_score_cells_sinr["alpha"] = geo_score_cells_sinr["scaled_geo_score"].apply(lambda x: 1 if x > 0.5 else map_range(x, 0., 0.5, 0., 1.))

geo_score_cells_sinr["red"] = geo_score_cells_sinr.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[0], axis=1)
geo_score_cells_sinr["grn"] = geo_score_cells_sinr.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[1], axis=1)
geo_score_cells_sinr["blu"] = geo_score_cells_sinr.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[2], axis=1)

In [201]:
view_state_sinr = pdk.ViewState(
    latitude=37.7749295, 
    longitude=-122.4194155, 
    zoom=3, 
    bearing=0, 
    pitch=0
)
layer_sinr = pdk.Layer(
    "H3HexagonLayer",
    geo_score_cells_sinr,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="[red * 255 * 0.3, grn * 255, blu * 255, alpha * 255]",
    get_line_color=[255, 100, 100],
    line_width_min_pixels=0,
)
r_sinr = pdk.Deck(
    layers=[layer_sinr], 
    initial_view_state=view_state_sinr,
    map_style=pdk.map_styles.LIGHT,
)
r_sinr.show()